# GEE Scripts — Python / `geemap` Port

Python translation of the Google Earth Engine JavaScript scripts in `../gee-scripts/`.
Uses the [`earthengine-api`](https://developers.google.com/earth-engine/guides/python_install)
and [`geemap`](https://geemap.org) packages.

**Original scripts:**
- `berm_export_Altar.js` — per-berm feature extraction + SAVI time-series export
- `flow_accumulation.js` — HydroSHEDS-based flow accumulation routing
- `bermModule.js` — shared geometry helpers

**Workflow:**
1. Authenticate & initialise GEE (cell 2)
2. Define AOI geometries (cell 3)
3. Load assets — DEM, berm structures, soils, roads (cell 4)
4. Geometry helpers — `get_poly`, `get_concav`, `get_angle` (cell 5)
5. Sentinel-2 SAVI collection (cell 6)
6. Per-berm summary extraction (`get_summary`) (cell 7)
7. Export summary table to Drive (cell 8)
8. SAVI time-series extraction & export (cell 9)
9. Flow accumulation (HydroSHEDS) (cell 10)
10. Map visualisation with `geemap` (cell 11)


In [16]:
# ── 1. Install / authenticate ────────────────────────────────────────────────
# Run once in a terminal if packages are not yet installed:
#   conda activate berms
#   pip install earthengine-api geemap
#
# Then authenticate (only needed once per machine):
#   earthengine authenticate

import ee
import geemap
import math
import pandas as pd

# Initialise with your GEE cloud project
ee.Initialize(project="ee-quatratavia")
print("GEE initialised ok")


GEE initialised ok


In [17]:
# ── 2. AOI geometries ────────────────────────────────────────────────────────
# Seven tiled AOI polygons covering the Altar Valley study area.
# Coordinates taken from the GEE import block in berm_export_Altar.js.

AltarValley1 = ee.Geometry.Polygon([[
    [-111.27318670654297, 31.969544168153206],
    [-111.31536027182821, 31.969544168153206],
    [-111.31639024008993, 32.07200719687609],
    [-111.37647172202352, 32.07200719687609],
]])

AltarValley2 = ee.Geometry.Polygon([[
    [-111.38901427073398, 31.969747519095876],
    [-111.40042975230136, 31.969529081164232],
    [-111.41038611216464, 31.96923782978016],
    [-111.49089529795566, 31.968218442659293],
    [-111.5035454272585,  31.97229592322598],
    [-111.52208485596944, 31.971422192637085],
    [-111.52431645386983, 31.947532669490183],
    [-111.53478779786397, 31.91344182816534],
    [-111.56860508912374, 31.908487406420182],
    [-111.57083668702413, 31.880212955857772],
    [-111.52380146973897, 31.879046812735556],
    [-111.40054860108663, 31.879921421461123],
    [-111.24379865753873, 31.8817528561191],
    [-111.24448530304655, 31.97150002187036],
]])

AltarValley3 = ee.Geometry.Polygon([[
    [-111.59930362351702, 31.879697489935594],
    [-111.59930362351702, 31.787672124490918],
    [-111.22164859422014, 31.787672124490918],
    [-111.22164859422014, 31.879697489935594],
]])

AltarValley4 = ee.Geometry.Polygon([[
    [-111.5908224903208, 31.78530898310566],
    [-111.5908224903208, 31.705750068031097],
    [-111.27290562020362, 31.705750068031097],
    [-111.27290562020362, 31.78530898310566],
]])

AltarValley5 = ee.Geometry.Polygon([[
    [-111.6031514673573, 31.70567717556796],
    [-111.6031514673573, 31.650748957536848],
    [-111.34874930671278, 31.650748957536848],
    [-111.34874930671278, 31.70567717556796],
]])

AltarValley6 = ee.Geometry.Polygon([[
    [-111.59524103250162, 31.65027027798726],
    [-111.59524103250162, 31.631636881374295],
    [-111.41739984597818, 31.631636881374295],
    [-111.41739984597818, 31.65027027798726],
]])

# Polygon to remove (edge artefact near south boundary)
remove_geom = ee.Geometry.Polygon([[
    [-111.49152757884167, 31.56577522324169],
    [-111.49163486720227, 31.564340013169563],
    [-111.48959638835095, 31.564349154704868],
    [-111.48965003253124, 31.565747799052403],
]])

AOI_TILES = {
    1: AltarValley1,
    2: AltarValley2,
    3: AltarValley3,
    4: AltarValley4,
    5: AltarValley5,
    6: AltarValley6,
}

# ── Choose which AOI tile to process ──────────────────────────────────────────
AOInum = 2   # ← change to 1-7
AOI = AOI_TILES[AOInum].difference(remove_geom, ee.ErrorMargin(1))
print(f"AOI {AOInum} set")


AOI 2 set


In [18]:
# ── 3. Load assets ───────────────────────────────────────────────────────────

# Berm structure polylines
structures = ee.FeatureCollection(
    "projects/ee-quatratavia/assets/berms/Altar050919/Structures_050919"
)

# Soil map units (for MUSYM / MUKEY lookup)
soils = ee.FeatureCollection(
    "projects/ee-quatratavia/assets/berms/20230721_AltarValleySoils_Shapefile"
)

# Roads (TIGER public dataset)
roads_fc = ee.FeatureCollection("TIGER/2016/Roads")

# Custom high-res DEM mosaic (DEP3 tiles)
asset_list = ee.data.listAssets("projects/ee-quatratavia/assets/berms/DEP3")["assets"]
asset_ids  = [a["id"] for a in asset_list]
combined_DEM = ee.ImageCollection(asset_ids).mosaic().rename("elevation")

# SRTM fallback DEM (30 m)
AOI_dem = ee.Image("USGS/SRTMGL1_003").clip(AOI.buffer(2000, 10)).select("elevation")
slope   = ee.Terrain.slope(AOI_dem)

# Precomputed SAVI image (GEE asset)
# SAVI_asset = ee.Image("projects/ee-quatratavia/assets/berms/AltarValley/Altar_SAVI")
SAVI_asset = ee.Image('projects/ee-quatratavia/assets/berms/AltarValley/Altar_SAVI_AugSep')

# Road distance raster (pixel-level, metres)
roads_img = ee.Image().paint(roads_fc).add(1)
rdist = (roads_img
         .fastDistanceTransform()
         .sqrt()
         .multiply(ee.Image.pixelArea().sqrt())
         .select([0], ["road_dist"]))

print("Assets loaded")


Assets loaded


In [19]:
# ── 4. Geometry helpers ───────────────────────────────────────────────────────
# Python translations of the JS helper functions in berm_export_Altar.js
# and bermModule.js.  All operate on ee.Feature / ee.Geometry objects.

# ── Shared parameters (mirror JS defaults) ────────────────────────────────────
POLY_END   = 45    # metres: perpendicular distance for upslope/downslope polys
POLY_WIDTH = 30    # metres: width of sampling polygon
BG_WIDTH   = 600   # metres: background buffer radius
OTHER_BUFFER = 200 # metres: buffer around neighbouring berms to exclude
RD_BUFFER    = 100 # metres: road exclusion distance


def _get_lat(lat1, angle, dist):
    """Spherical destination latitude (radians inputs, degrees output)."""
    return ee.Number.expression(
        "rad2deg*asin(sin(lat1)*cos(d) + cos(lat1)*sin(d)*cos(a))",
        {
            "lat1":    lat1.multiply(math.pi).divide(180),
            "rad2deg": ee.Number(180).divide(math.pi),
            "d":       dist,
            "a":       angle,
        },
    )


def _get_lon(lon1, lat1, lat_new, angle, dist):
    """Spherical destination longitude."""
    return ee.Number.expression(
        "rad2deg*(lon1 + atan((sin(a)*sin(d)*cos(lat1))/(cos(d)-sin(lat1)*sin(lat2))))",
        {
            "lon1":    lon1.multiply(math.pi).divide(180),
            "lat1":    lat1.multiply(math.pi).divide(180),
            "lat2":    lat_new.multiply(math.pi).divide(180),
            "rad2deg": ee.Number(180).divide(math.pi),
            "d":       dist,
            "a":       angle,
        },
    )


def get_poly(berm, poly_end, poly_width, rotate):
    """
    Generate a parallelogram polygon perpendicular to the berm at one end.
    rotate=0 → one side; rotate=pi → the other.
    Mirrors JS get_poly() in berm_export_Altar.js.
    """
    rotate = ee.Number(rotate)
    coords = berm.geometry().coordinates()
    p1 = ee.List(coords.get(0))
    p2 = ee.List(coords.get(-1))

    lon1 = ee.Number(p1.get(0));  lat1 = ee.Number(p1.get(-1))
    lon2 = ee.Number(p2.get(0));  lat2 = ee.Number(p2.get(-1))

    slope = (lat2.subtract(lat1)).divide(lon2.subtract(lon1))
    angle = ee.Number(slope).atan().multiply(-1).add(rotate)

    dist   = ee.Number(poly_end).divide(6371000)
    buffer = ee.Number(poly_end).subtract(poly_width).divide(6371000)

    lat3 = _get_lat(lat1, angle, dist);   lat4 = _get_lat(lat2, angle, dist)
    lon3 = _get_lon(lon1, lat1, lat3, angle, dist)
    lon4 = _get_lon(lon2, lat2, lat4, angle, dist)

    lat11 = _get_lat(lat1, angle, buffer); lat22 = _get_lat(lat2, angle, buffer)
    lon11 = _get_lon(lon1, lat1, lat3, angle, buffer)
    lon22 = _get_lon(lon2, lat2, lat4, angle, buffer)

    p1b = ee.Geometry.Point([lon11, lat11])
    p2b = ee.Geometry.Point([lon22, lat22])
    p3  = ee.Geometry.Point([lon3,  lat3])
    p4  = ee.Geometry.Point([lon4,  lat4])

    return ee.Geometry.Polygon([p1b, p2b, p4, p3]).buffer(3, 1)


def get_angle(berm):
    """Bearing of the berm line in degrees."""
    coords = berm.geometry().coordinates()
    p1 = ee.List(coords.get(0));  p2 = ee.List(coords.get(-1))
    lon1 = ee.Number(p1.get(0));  lat1 = ee.Number(p1.get(-1))
    lon2 = ee.Number(p2.get(0));  lat2 = ee.Number(p2.get(-1))
    slope = (lat2.subtract(lat1)).divide(lon2.subtract(lon1))
    angle = ee.Number(slope).atan().multiply(-1)
    return angle.divide(ee.Number(math.pi).divide(180))


def reduce_elevation(feat, dem=None):
    """Median elevation (SRTM) over a geometry."""
    if dem is None:
        dem = AOI_dem
    return dem.reduceRegion(
        reducer=ee.Reducer.median(),
        geometry=feat,
        scale=1,
        maxPixels=1e9,
    ).get("elevation")


def reduce_elevation_mosaic(feat):
    """Median elevation (custom DEM mosaic) over a geometry."""
    return combined_DEM.reduceRegion(
        reducer=ee.Reducer.median(),
        geometry=feat,
        scale=1,
        maxPixels=1e9,
    ).get("elevation")


def get_others(berm):
    """Other berm structures within BG_WIDTH of this berm."""
    others = structures.filterBounds(berm.geometry().buffer(BG_WIDTH))
    return others.filter(ee.Filter.neq("Shape_Leng", berm.get("Shape_Leng")))


def get_other_distance(berm, others):
    """Median pixel distance from this berm to neighbouring berms."""
    other_paint = ee.Image().paint(others).add(1)
    other_dist  = (other_paint.fastDistanceTransform().sqrt()
                   .multiply(ee.Image.pixelArea().sqrt())
                   .select([0], ["others"])
                   .clip(berm.geometry().buffer(BG_WIDTH)))
    return other_dist.reduceRegion(
        reducer=ee.Reducer.median(),
        geometry=berm.geometry(),
        scale=1,
    ).get("others")


print("Geometry helpers defined")


Geometry helpers defined


In [20]:
# ── 5. Sentinel-2 SAVI collection ─────────────────────────────────────────────

def add_savi(image):
    """Add a SAVI band: ((NIR - Red) / (NIR + Red + 0.5)) * 1.5"""
    savi = image.expression(
        "((NIR - Red) / (NIR + Red + 0.5)) * 1.5",
        {
            "NIR": image.select("B8").multiply(0.0001),
            "Red": image.select("B4").multiply(0.0001),
        },
    ).rename("savi")
    return image.addBands(savi)


def mask_s2_clouds(image):
    """Mask clouds using the Sentinel-2 QA60 band."""
    qa = image.select("QA60")
    cloud_bit_mask  = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = (qa.bitwiseAnd(cloud_bit_mask).eq(0)
              .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0)))
    return image.updateMask(mask).divide(10000)


S2 = (ee.ImageCollection("COPERNICUS/S2")
        .filterMetadata("CLOUDY_PIXEL_PERCENTAGE", "less_than", 1)
        .map(mask_s2_clouds)
        .map(add_savi))

savi_S2 = S2.select("savi")

# Percentile reducer used throughout (mirrors JS percentileReducer)
PERCENTILES       = [10, 25, 50, 75, 90]
percentile_reducer = ee.Reducer.percentile(PERCENTILES)

print("Sentinel-2 SAVI collection ready")


Sentinel-2 SAVI collection ready


In [21]:
# ── 6. Per-berm summary extraction ───────────────────────────────────────────
# Python translation of summarize_upslope_downslope() + get_summary()
# from berm_export_Altar.js.

def summarize_upslope_downslope(berm, poly_end=POLY_END):
    """
    For a single berm feature, derive upslope and downslope sampling polygons,
    compute elevation, SAVI, and background SAVI statistics.
    Returns an ee.Dictionary with keys like zU_45, saviU_45, savi_background, …
    """
    poly1 = get_poly(berm, poly_end, POLY_WIDTH, 0)
    poly2 = get_poly(berm, poly_end, POLY_WIDTH, math.pi)

    z1 = reduce_elevation(poly1)
    z2 = reduce_elevation(poly2)
    z1m = reduce_elevation_mosaic(poly1)
    z2m = reduce_elevation_mosaic(poly2)

    compare  = ee.Number.expression("z1 > z2", {"z1": z1,  "z2": z2})
    compareM = ee.Number.expression("z1 > z2", {"z1": z1m, "z2": z2m})
    flag = ee.Algorithms.IsEqual(compare, 1)

    polyU = ee.Algorithms.If(flag, poly1, poly2)
    polyD = ee.Algorithms.If(flag, poly2, poly1)
    zU    = ee.Algorithms.If(flag, z1, z2)
    zD    = ee.Algorithms.If(flag, z2, z1)

    im = savi_S2.median()

    saviU = im.reduceRegion(
        reducer=ee.Reducer.median(), geometry=polyU, scale=10
    )
    saviD = im.reduceRegion(
        reducer=ee.Reducer.median(), geometry=polyD, scale=10
    )

    # Background SAVI: large buffer around berm, excluding nearby structures
    # and roads
    big  = berm.geometry().buffer(BG_WIDTH, 10)
    mask = structures.filterBounds(big).geometry().buffer(OTHER_BUFFER, 10)
    clip = big.difference(mask, ee.ErrorMargin(1))

    savi_background = (im.clip(clip)
                         .updateMask(rdist.gt(RD_BUFFER))
                         .reduceRegion(
                             reducer=ee.Reducer.median(),
                             geometry=clip,
                             maxPixels=1e9,
                             scale=10)
                         .get("savi"))

    dist = poly_end
    keys = [
        f"zU_{dist}", f"zD_{dist}",
        f"saviU_{dist}", f"saviD_{dist}",
        "savi_background",
        "compare", "compareM",
    ]
    values = [
        zU, zD,
        saviU.get("savi"), saviD.get("savi"),
        savi_background,
        compare, compareM,
    ]
    return ee.Dictionary.fromLists(keys, values)


def get_summary(berm):
    """
    Full per-berm feature extraction.
    Returns an ee.Feature with upslope/downslope SAVI, elevation,
    soil map unit, road distance, and neighbouring-berm distance.
    """
    berm_angle = get_angle(berm)
    savi_dict  = summarize_upslope_downslope(berm, POLY_END)

    road_dist = rdist.reduceRegion(
        reducer=ee.Reducer.median(),
        geometry=berm.geometry(),
        scale=10,
    ).get("road_dist")

    soil       = soils.filterBounds(berm.geometry()).first()
    others     = get_others(berm)
    other_dist = get_other_distance(berm, others)
    other_elev = reduce_elevation(berm.geometry())
    berm_elev  = reduce_elevation(others.geometry())
    centroid   = berm.geometry().centroid().coordinates()

    props = ee.Dictionary({
        "road_dist":  road_dist,
        "other_dist": other_dist,
        "other_elev": other_elev,
        "berm_elev":  berm_elev,
        "longitude":  centroid.get(0),
        "latitude":   centroid.get(1),
        "berm_angle": berm_angle,
        "MUSYM":      soil.get("MUSYM"),
        "MUKEY":      soil.get("MUKEY"),
        "AREASYMBOL": soil.get("AREASYMBOL"),
        "id":         ee.Feature(berm).id(),
    })

    return (ee.Feature(berm.geometry(), props.combine(savi_dict))
              .copyProperties(berm))


print("get_summary() defined")


get_summary() defined


In [22]:
# ── 7. Export summary table to Drive ─────────────────────────────────────────
# Applies get_summary() to every berm in the chosen AOI tile and exports to
# Google Drive as a CSV (folder: berm_exports).
# This mirrors the first Export.table.toDrive() call in berm_export_Altar.js.

s = (structures
     .filterBounds(AOI)
     .filter(ee.Filter.gt("Shape_Leng", 0))
     .map(get_summary))

export_task = ee.batch.Export.table.toDrive(
    collection=s,
    folder="berm_exports",
    description=f"Altar_{AOInum}",
    fileFormat="CSV",
)

export_task.start()
print(f"Export task started: Altar_{AOInum}  (status: {export_task.status()['state']})")
print("Monitor at: https://code.earthengine.google.com/tasks")


Export task started: Altar_2  (status: READY)
Monitor at: https://code.earthengine.google.com/tasks


In [23]:
# ── 8. SAVI time-series extraction & export ───────────────────────────────────
# For each berm in the AOI, extracts a per-image SAVI record including
# upslope, downslope, and background statistics.
# Mirrors timeSeriesForBerm() + extractSAVI() + Export.table in berm_export_Altar.js.

def extract_savi(image, berm):
    """
    Extract per-image SAVI statistics for one berm:
    upslope polygon, downslope polygon, and background.
    """
    savi = image.select("savi")

    poly1 = get_poly(berm, POLY_END, POLY_WIDTH, 0)
    poly2 = get_poly(berm, POLY_END, POLY_WIDTH, math.pi)

    z1 = reduce_elevation_mosaic(poly1)
    z2 = reduce_elevation_mosaic(poly2)
    compare = ee.Number.expression("z1 > z2", {"z1": z1, "z2": z2})
    flag    = ee.Algorithms.IsEqual(compare, 1)

    polyU = ee.Algorithms.If(flag, poly1, poly2)
    polyD = ee.Algorithms.If(flag, poly2, poly1)

    saviU = savi.reduceRegion(
        reducer=percentile_reducer, geometry=polyU, scale=10
    )
    saviD = savi.reduceRegion(
        reducer=percentile_reducer, geometry=polyD, scale=10
    )

    # Background: buffer minus nearby berms and roads
    big  = berm.geometry().buffer(BG_WIDTH, 10)
    mask = structures.filterBounds(big).geometry().buffer(OTHER_BUFFER, 10)
    clip = big.difference(mask, ee.ErrorMargin(1))

    savi_background = (savi.clip(clip)
                           .updateMask(rdist.gt(RD_BUFFER))
                           .reduceRegion(
                               reducer=percentile_reducer,
                               geometry=berm.geometry().buffer(1000, 10),
                               scale=10))

    centroid = berm.geometry().centroid().coordinates()

    return (ee.Feature(None, {
        "saviU":      saviU,
        "saviD":      saviD,
        "background": savi_background,
        "date":       image.date(),
        "longitude":  centroid.get(0),
        "latitude":   centroid.get(1),
    }).copyProperties(berm))


def time_series_for_berm(berm):
    return (savi_S2
            .filterBounds(berm.geometry().buffer(1000, 10))
            .map(lambda img: extract_savi(img, berm)))


s_timeseries = (structures
                .filterBounds(AOI)
                .map(time_series_for_berm)
                .flatten())

ts_task = ee.batch.Export.table.toDrive(
    collection=s_timeseries,
    folder="berm_exports",
    description=f"Altar_timeseries_{AOInum}",
    fileFormat="CSV",
)

ts_task.start()
print(f"Time-series export started: Altar_timeseries_{AOInum}  "
      f"(status: {ts_task.status()['state']})")


Time-series export started: Altar_timeseries_2  (status: READY)


In [24]:
# ── 9. Flow accumulation (HydroSHEDS D8) ─────────────────────────────────────
# Python translation of flow_accumulation.js.
# Routes a data image (default: pixel count = 1) downstream using the
# WWF HydroSHEDS 30-arc-second flow-direction grid.

# ── Parameters ────────────────────────────────────────────────────────────────
ITERATION_STEPS = 100        # number of downstream routing iterations
DATA_IMAGE      = ee.Image(1)   # what to route — change to e.g. ee.Image.pixelArea()
BANDNAME        = "constant"    # band in DATA_IMAGE
UNIT            = "pixels"      # label only
OUTPUT_TYPE     = "uint32"

flow_dir = ee.Image("WWF/HydroSHEDS/30DIR").select(["b1"])

# ── D8 direction lookup (dir_id matches HydroSHEDS encoding) ─────────────────
DIRECTIONS = [
    {"direction": "E",  "dir_id": 1,   "weight": [[0,0,0],[1,0,0],[0,0,0]]},
    {"direction": "SE", "dir_id": 2,   "weight": [[1,0,0],[0,0,0],[0,0,0]]},
    {"direction": "S",  "dir_id": 4,   "weight": [[0,1,0],[0,0,0],[0,0,0]]},
    {"direction": "SW", "dir_id": 8,   "weight": [[0,0,1],[0,0,0],[0,0,0]]},
    {"direction": "W",  "dir_id": 16,  "weight": [[0,0,0],[0,0,1],[0,0,0]]},
    {"direction": "NW", "dir_id": 32,  "weight": [[0,0,0],[0,0,0],[0,0,1]]},
    {"direction": "N",  "dir_id": 64,  "weight": [[0,0,0],[0,0,0],[0,1,0]]},
    {"direction": "NE", "dir_id": 128, "weight": [[0,0,0],[0,0,0],[1,0,0]]},
]
dir_fc = ee.FeatureCollection([
    ee.Feature(None, {"weight": d["weight"], "dir_id": d["dir_id"]})
    for d in DIRECTIONS
])

# ── Routing iteration function ────────────────────────────────────────────────
def iterate_route(iter_step, data):
    data = ee.Image(data)

    def route(ft):
        kernel   = ee.Kernel.fixed(width=3, height=3, weights=ft.get("weight"))
        dir_id   = ee.Number(ft.get("dir_id"))
        routed   = data.select("rout").updateMask(flow_dir.eq(dir_id))
        return routed.reduceNeighborhood(
            reducer=ee.Reducer.sum(), kernel=kernel, skipMasked=False
        )

    step = (ee.ImageCollection(dir_fc.map(route))
              .sum()
              .rename("rout")
              .reproject(flow_dir.projection())
              .set("iter_idx", iter_step)
              .clip(AOI))

    summed = step.select("rout").add(data.select("summed")).rename("summed")
    return step.addBands(summed)


# ── Initialise and run iterations ─────────────────────────────────────────────
data_init = (DATA_IMAGE
             .select([BANDNAME], ["rout"])
             .addBands(DATA_IMAGE.select([BANDNAME], ["summed"]))
             .set("iter_idx", 0))

steps     = ee.List.sequence(1, ITERATION_STEPS)
flow_full = ee.Image(steps.iterate(iterate_route, data_init)).cast(
    {"rout": OUTPUT_TYPE, "summed": OUTPUT_TYPE}
)

print("Flow accumulation image defined — add to map or export below")
print("  flow_full.select('summed')  — cumulative upstream area")


Flow accumulation image defined — add to map or export below
  flow_full.select('summed')  — cumulative upstream area


In [25]:
# ── 10. Map visualisation with geemap ─────────────────────────────────────────

Map = geemap.Map()
Map.centerObject(AOI, 12)

# Satellite basemap
Map.add_basemap("SATELLITE")

# SRTM hillshade
hillshade = ee.Terrain.hillshade(AOI_dem)
Map.addLayer(hillshade, {"min": 0, "max": 255}, "Hillshade", opacity=0.4)

# SAVI median composite
savi_median = savi_S2.median().clip(AOI)
Map.addLayer(
    savi_median,
    {"min": 0, "max": 0.8, "palette": ["#d73027","#fee090","#91cf60","#1a9850"]},
    "SAVI median",
)

# Flow accumulation (log scale for visibility)
flow_vis = flow_full.select("summed").log().clip(AOI)
Map.addLayer(
    flow_vis,
    {"min": 0, "max": 10, "palette": ["white", "#2166ac"]},
    "Flow accumulation (log)",
)

# Berm structures
Map.addLayer(
    structures.filterBounds(AOI),
    {"color": "orange"},
    "Berm structures",
)

# AOI boundary
Map.addLayer(ee.Image().paint(AOI, 1, 2), {"palette": "red"}, "AOI boundary")

Map
# TODO: update SAVI location. 
# Want to be able to click on a berm and see the SAVI time series for that berm.

Map(center=[31.923527338421746, -111.39404693332962], controls=(WidgetControl(options=['position', 'transparen…

In [26]:
# ── 11. Recompute SAVI from Sentinel-2 (Aug–Sep) → export as GEE asset ───────
# Filters to August–September across all available years to capture
# monsoon-season peak greenness.  Exports a 3-band image (median, p10, p90)
# to a GEE asset so it can be loaded in future cells via SAVI_asset.
#
# After the task finishes (~10–30 min), update SAVI_asset in cell 4 to:
#   ee.Image("projects/ee-quatratavia/assets/berms/AltarValley/Altar_SAVI_AugSep")

# ── Full study-area AOI (union of all tiles) ──────────────────────────────────
_full_aoi = ee.Geometry.MultiPolygon([
    AltarValley1, AltarValley2, AltarValley3,
    AltarValley4, AltarValley5, AltarValley6,
]).difference(remove_geom, ee.ErrorMargin(1))

# ── August–September composite ────────────────────────────────────────────────
_aug_sep_filter = ee.Filter.calendarRange(8, 9, "month")
_savi_aug_sep   = savi_S2.filter(_aug_sep_filter)

_savi_export_img = (
    _savi_aug_sep.median().rename("savi_median")
    .addBands(_savi_aug_sep.reduce(ee.Reducer.percentile([10])).rename("savi_p10"))
    .addBands(_savi_aug_sep.reduce(ee.Reducer.percentile([90])).rename("savi_p90"))
    .clip(_full_aoi)
)

# ── Export to asset ───────────────────────────────────────────────────────────
_asset_id = "projects/ee-quatratavia/assets/berms/AltarValley/Altar_SAVI_AugSep"

savi_asset_task = ee.batch.Export.image.toAsset(
    image=_savi_export_img,
    description="Altar_SAVI_AugSep",
    assetId=_asset_id,
    region=_full_aoi,
    scale=10,
    maxPixels=1e13,
)

savi_asset_task.start()
print(f"SAVI asset export started  →  {_asset_id}")
print(f"Status: {savi_asset_task.status()['state']}")
print("Monitor at: https://code.earthengine.google.com/tasks")


SAVI asset export started  →  projects/ee-quatratavia/assets/berms/AltarValley/Altar_SAVI_AugSep
Status: READY
Monitor at: https://code.earthengine.google.com/tasks
